Настройка окружения и путей

In [1]:
import os
import sys
from IPython.display import display, Markdown

# Настраиваем абсолютные пути, чтобы Jupyter увидел папку src и .env
base_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(os.path.join(base_dir, 'src'))

from dotenv import load_dotenv
from langchain_community.llms import YandexGPT
from rag.document_loader import DocumentLoader
from rag.chunking import ChunkingStrategy
from rag.vector_store import VectorStoreManager
from main import VideoSurveillanceRAG

# Загружаем ключи
load_dotenv(os.path.join(base_dir, '.env'))
print("✅ Библиотеки и пути загружены!")

C:\Users\danil\PycharmProjects\ai-course-labs\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


✅ Библиотеки и пути загружены!


Инициализация RAG-базы (Под капотом)

In [2]:
print("⏳ Инициализация RAG системы...")
docs_dir = os.path.join(base_dir, "data", "documents")
chroma_dir = os.path.join(base_dir, "data", "chroma_db")

# 1. Загрузка и чанкинг
loader = DocumentLoader(source_directory=docs_dir)
docs = loader.load_directory()
chunks = ChunkingStrategy.split_documents(docs)
print(f"📄 Документы загружены и разбиты на {len(chunks)} чанков.")

# 2. Векторная БД (Chroma)
vectorstore = VectorStoreManager(persist_directory=chroma_dir)
vectorstore.add_documents(chunks)
print("🗄️ Векторная база данных обновлена.")

# 3. Подключение Яндекса и запуск RAG
llm = YandexGPT(
    iam_token=os.getenv("YANDEX_IAM_TOKEN"),
    folder_id=os.getenv("YANDEX_FOLDER_ID"),
    temperature=0.1
)
rag = VideoSurveillanceRAG(vectorstore=vectorstore, llm=llm)
print("🚀 ИИ-ассистент готов к ответам на вопросы!")

⏳ Инициализация RAG системы...
📄 Документы загружены и разбиты на 3 чанков.


C:\Users\danil\PycharmProjects\ai-course-labs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6632.97it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


🗄️ Векторная база данных обновлена.
🚀 ИИ-ассистент готов к ответам на вопросы!


Интерактивное тестирование (Вывод)

In [3]:
def ask_agent(question):
    res = rag.query(question)
    md_text = f"### ❓ Вопрос: {question}\n\n**🤖 Ответ ИИ:** {res['answer']}\n\n*⏱ Время ответа: {res['execution_time']}с | 📚 Использовано источников: {len(res['sources'])}*"
    display(Markdown(md_text))
    print("-" * 60)

# Тестируем нашу базу знаний
ask_agent("Что делать, если в зоне Б (Парковка) машина стоит 2 дня?")
ask_agent("Можно ли людям находиться в Зоне А ночью?")
ask_agent("Как система распознает номера машин?")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


### ❓ Вопрос: Что делать, если в зоне Б (Парковка) машина стоит 2 дня?

**🤖 Ответ ИИ:** Согласно регламенту, если автомобиль находится без движения более 24 часов, генерируется предупреждение.

*⏱ Время ответа: 0.83с | 📚 Использовано источников: 3*

------------------------------------------------------------


### ❓ Вопрос: Можно ли людям находиться в Зоне А ночью?

**🤖 Ответ ИИ:** Согласно регламенту, в Зоне А (Периметр) категорически запрещено нахождение людей с 22:00 до 06:00. При обнаружении генерируется критический инцидент.

*⏱ Время ответа: 1.13с | 📚 Использовано источников: 3*

------------------------------------------------------------


### ❓ Вопрос: Как система распознает номера машин?

**🤖 Ответ ИИ:** Согласно регламенту, информации об этом нет.

*⏱ Время ответа: 0.57с | 📚 Использовано источников: 3*

------------------------------------------------------------
